# Tunix-Med: Baseline Medical Knowledge Evaluation

This notebook measures how well the **pre-trained `google/gemma-3-1b-it` model** answers a set of 25 medical questions based on cardiology clinical guidelines **before any fine-tuning**. The scores produced here become the **baseline** we compare against after domain-specific training with **Tunix**.

### Evaluation pipeline

For each question the notebook:
1. Generates an answer using the pre-trained model.
2. Computes **perplexity** of the reference answer.
3. Scores the answer with three independent methods:
   - **Keyword matching**
   - **Semantic similarity** (Sentence-BERT)
   - **AI judge** (LLM-as-a-Judge)
4. Aggregates results by question difficulty.

In [ ]:
import os
import gc
import warnings
import torch
import numpy as np
import pandas as pd
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer
from huggingface_hub import login

def init():
    os.environ["TOKENIZERS_PARALLELISM"] = "false"
    token = os.environ.get("HF_TOKEN")
    if token:
        login(token=token)
    torch.cuda.empty_cache()
    gc.collect()
    warnings.filterwarnings("ignore")

def info_device():
    if torch.backends.mps.is_available():
        device = torch.device("mps")
    elif torch.cuda.is_available():
        device = torch.device("cuda")
    else:
        device = torch.device("cpu")
    print(f"Using device: {device}")
    return device

def is_bfloat16_supported():
    if torch.backends.mps.is_available():
        return True
    return torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8

init()
device = info_device()
dtype = torch.bfloat16 if is_bfloat16_supported() else torch.float16
print(f"Using dtype: {dtype}")

## Medical Evaluation Dataset
We use a curated set of 25 questions based on cardiology clinical guidelines (e.g., ESC/ACC/AHA).

In [ ]:
medical_qa = [
    {"Question": "What is the recommended first-line treatment for uncomplicated hypertension according to ESC guidelines?", "Answer": "ACE inhibitors or ARBs, combined with a calcium channel blocker or a thiazide/thiazide-like diuretic.", "Difficulty": "Easy"},
    {"Question": "In patients with heart failure with reduced ejection fraction (HFrEF), what are the 'four pillars' of medical therapy?", "Answer": "ACEi/ARNI, beta-blockers, MRA (mineralocorticoid receptor antagonists), and SGLT2 inhibitors.", "Difficulty": "Easy"},
    {"Question": "What is the definition of Stage 1 hypertension according to the 2017 ACC/AHA guidelines?", "Answer": "Systolic blood pressure 130-139 mmHg or diastolic blood pressure 80-89 mmHg.", "Difficulty": "Medium"},
    {"Question": "What is the target LDL-C level for patients at very high cardiovascular risk according to the 2019 ESC/EAS guidelines?", "Answer": "< 1.4 mmol/L (55 mg/dL) and a reduction of >= 50% from baseline.", "Difficulty": "Medium"},
    {"Question": "In the context of acute coronary syndrome, what does the 'TIMI score' predict?", "Answer": "The risk of death and ischemic events in patients with UA/NSTEMI or STEMI.", "Difficulty": "Hard"}
    # ... (more questions would be added here to reach 25)
]

data = pd.DataFrame(medical_qa)
data

## Loading the Model

In [ ]:
MODEL_NAME = "google/gemma-3-1b-it"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=dtype,
    device_map=device,
    use_cache=True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

## Baseline Evaluation Loop

In [ ]:
results_list = []
instructions = "\nProvide a concise medical answer based on clinical guidelines."

model.eval()
for index, row in tqdm(data.iterrows(), total=len(data)):
    question = row["Question"]
    answer = row["Answer"]
    difficulty = row["Difficulty"]

    inputs = tokenizer.apply_chat_template(
        [{"role": "user", "content": question + instructions}],
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(
            inputs,
            max_new_tokens=256,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
        generated_token_ids = outputs[0, inputs.shape[-1] :]
        generated_text = tokenizer.decode(generated_token_ids, skip_special_tokens=True).strip()

        # Perplexity calculation
        reference_ids = tokenizer(answer, return_tensors="pt", add_special_tokens=False).input_ids.to(device)
        full_input_ids = torch.cat([inputs, reference_ids], dim=1)
        labels = full_input_ids.clone()
        labels[:, : inputs.shape[1]] = -100
        loss = model(full_input_ids, labels=labels).loss
        perplexity = torch.exp(loss).item()

    results_list.append({
        "question": question,
        "expected_answer": answer,
        "generated_answer": generated_text,
        "difficulty": difficulty,
        "perplexity": perplexity,
    })

results_df = pd.DataFrame(results_list)
results_df

## Scoring and Results Visualization
Similar to the Sherlock workshop, we would now implement keyword matching, semantic similarity, and AI judge scoring here.